In [62]:
from pprint import pprint
import pathlib
import csv
import os
from utils import *
import csv
import contractions
import shutil
import emoji
from emot.emo_unicode import EMOTICONS_EMO
import re

In [63]:
# create preprocessed dir if not exists
pathlib.Path(preprocessed_data_path).mkdir(parents=True, exist_ok=True)

# copy raw reviews for preprocessing
shutil.copytree(raw_data_path, preprocessed_data_path, dirs_exist_ok=True)

def preprocess_data(process_fn):
    for name in os.listdir(preprocessed_data_path):
        path = os.path.join(preprocessed_data_path, name)

        with open(path, newline="") as f:
            reader = csv.DictReader(f)
            data = list(reader)

            data = process_fn(data)

            write_path = os.path.join(preprocessed_data_path, name)
            file = open(write_path, "w")
            output_csv(data, file)

            file.close()

In [64]:
contractions_dict = {
    "i'mma": "i will"
}

def expand_contractions(data):
    for review in data:
        expanded_words = []
        content_words = review["content"].split(" ")
        for word in content_words:
            word_new = ""
            if word not in contractions_dict.keys():
                word_new = contractions.fix(word)
            else:
                word_new = contractions_dict[word]
            expanded_words.append(word_new)
        review["content"] = " ".join(expanded_words)

    return data

preprocess_data(expand_contractions)

In [65]:
def remove_emojis(data):
    for review in data:
        text = emoji.replace_emoji(review["content"], replace="")

        review["content"] = text
    return data
preprocess_data(remove_emojis)

In [73]:
emoticons_dict_custom = EMOTICONS_EMO
emoticons_dict_custom["¯\\_(ツ)_/¯"] = "Shrug"

emoticon_regex = re.compile(
    "|".join(map(re.escape, emoticons_dict_custom.keys()))
)

def remove_emoticons(data):
    for review in data:
        text = re.sub(emoticon_regex, "", review["content"])

        review["content"] = text
    return data
preprocess_data(remove_emoticons)

In [67]:
def remove_stopwords(data):
    for review in data:
        content_words = review["content"].split(" ")
        filtered = [w for w in content_words if w not in STOPWORDS]

        review["content"] = " ".join(filtered)
    return data

preprocess_data(remove_stopwords)